# BA-CVA walkthrough

This notebook builds synthetic BA-CVA counterparty and netting-set inputs, runs reduced BA-CVA, then adds hedge records for the full BA-CVA path.


In [ ]:
from pathlib import Path
import sys

HERE = Path.cwd()
PACKAGE_ROOT = None
REPO_ROOT = None

for candidate in (HERE, *HERE.parents):
    if (candidate / "src" / "frtb_cva").exists():
        PACKAGE_ROOT = candidate
        REPO_ROOT = candidate.parent.parent if candidate.parent.name == "packages" else candidate
        break
    package_candidate = candidate / "packages" / "frtb-cva"
    if (package_candidate / "src" / "frtb_cva").exists():
        REPO_ROOT = candidate
        PACKAGE_ROOT = package_candidate
        break

if PACKAGE_ROOT is None or REPO_ROOT is None:
    raise RuntimeError("Could not locate the frtb-cva package root")

workspace_src_paths = tuple(sorted((REPO_ROOT / "packages").glob("*/src")))
for path in (
    PACKAGE_ROOT / "examples",
    PACKAGE_ROOT,
    PACKAGE_ROOT / "src",
    *workspace_src_paths,
    REPO_ROOT,
):
    text = str(path)
    if text not in sys.path:
        sys.path.insert(0, text)

try:
    from IPython.display import Markdown, display
except ImportError:

    class Markdown(str):
        pass

    def display(value: object) -> None:
        print(value)


PACKAGE_ROOT

In [ ]:
from cva_notebook_data import (
    markdown_table,
    notebook_context,
    sample_counterparties,
    sample_direct_hedge,
    sample_ineligible_hedge,
    sample_netting_sets,
)
from frtb_cva import CvaMethod, calculate_cva_capital, validate_cva_result_reconciliation

counterparties = sample_counterparties()
netting_sets = sample_netting_sets(counterparties)
reduced_context = notebook_context(
    method=CvaMethod.BA_CVA_REDUCED, run_id="cva-ba-reduced-notebook"
)
reduced_result = calculate_cva_capital(reduced_context, counterparties, netting_sets)
validate_cva_result_reconciliation(reduced_result)

display(
    Markdown(
        markdown_table(
            ("metric", "value"),
            (
                ("total CVA capital", f"{reduced_result.total_cva_capital:,.2f}"),
                ("counterparty capital records", len(reduced_result.ba_cva_counterparty_capitals)),
                ("netting-set audit lines", len(reduced_result.ba_cva_netting_set_lines)),
                ("input hash", reduced_result.input_hash[:16]),
            ),
        )
    )
)

In [ ]:
line_rows = tuple(
    (
        line.netting_set_id,
        line.counterparty_id,
        line.sector.value,
        line.credit_quality.value,
        f"{line.ead:,.0f}",
        f"{line.standalone_capital:,.2f}",
    )
    for line in reduced_result.ba_cva_netting_set_lines
)

display(
    Markdown(
        markdown_table(
            ("netting set", "counterparty", "sector", "quality", "EAD", "stand-alone capital"),
            line_rows,
        )
    )
)

In [ ]:
hedges = (sample_direct_hedge(), sample_ineligible_hedge())
full_context = notebook_context(method=CvaMethod.BA_CVA_FULL, run_id="cva-ba-full-notebook")
full_result = calculate_cva_capital(full_context, counterparties, netting_sets, hedges=hedges)
validate_cva_result_reconciliation(full_result)
assert full_result.ba_cva_full is not None

display(
    Markdown(
        markdown_table(
            ("metric", "value"),
            (
                ("reduced BA-CVA", f"{full_result.ba_cva_full.k_reduced:,.2f}"),
                ("full BA-CVA", f"{full_result.ba_cva_full.k_full:,.2f}"),
                ("hedged portfolio term", f"{full_result.ba_cva_full.k_portfolio_hedged:,.2f}"),
                ("eligible hedge records", len(full_result.ba_cva_full.hedge_lines)),
            ),
        )
    )
)

In [ ]:
hedge_rows = tuple(
    (
        line.hedge_id,
        line.counterparty_id,
        line.hedge_type.value,
        line.eligibility.value,
        line.reference_relation.value,
        line.reason_code,
        f"{line.snh_contribution:,.2f}",
    )
    for line in full_result.ba_cva_full.hedge_lines
)

display(
    Markdown(
        markdown_table(
            (
                "hedge",
                "counterparty",
                "type",
                "eligibility",
                "relation",
                "reason",
                "SNH contribution",
            ),
            hedge_rows,
        )
    )
)